# 🎬 Clasificación de reseñas de películas con NLP

Este notebook presenta el flujo completo para clasificar reseñas de películas
utilizando técnicas de procesamiento de lenguaje natural (NLP).

In [ ]:
import pandas as pd
import numpy as np

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


## 📌 Carga de datos

In [ ]:
df = pd.read_csv("data.csv")  # ajusta el nombre real del archivo
df.head()


## 🧹 Preprocesamiento de texto

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return " ".join(
        lemmatizer.lemmatize(w)
        for w in tokens
        if w.isalpha() and w not in stop_words
    )

df["text_clean"] = df["review"].apply(preprocess_text)


## 🔢 Vectorización TF-IDF

In [ ]:
X = df["text_clean"]
y = df["sentiment"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=12345
)

vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_valid_tfidf = vectorizer.transform(X_valid)


## 🤖 Entrenamiento del modelo

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)


## 📊 Evaluación

In [ ]:
y_pred_proba = model.predict_proba(X_valid_tfidf)[:, 1]
roc_auc_score(y_valid, y_pred_proba)
